# Fiche 6 — Découper un pipeline en étapes et le piloter par un fichier de configuration

| | |
|---|---|
| 🎯 **Objectif** | Séparer préparation, entraînement et évaluation, centraliser les réglages dans un YAML et produire des erreurs claires |
| ⏱️ **Durée** | 40 min |
| 🧰 **Prérequis** | Fonctions Python |
| 🔗 **Dans le projet** | Étape 2 et pipeline MLOps (étapes automatisées, configuration centralisée) |

> **L'idée clé.** Un pipeline, ce sont des **étapes séparées** qui communiquent par des **fichiers** (une entrée, une sortie chacune), et **un seul endroit** pour les réglages. On peut alors relancer une étape sans tout refaire, et changer un réglage sans toucher au code.

## 1. Un fichier de configuration YAML

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
import yaml

texte = '''
graine: 42   # fixe le hasard : mêmes résultats à chaque exécution
donnees:
  entree: donnees_brutes.csv
  sortie: donnees_propres.csv
modele:
  nom: ridge
  alpha: 1.0
'''
config = yaml.safe_load(texte)
print(config)
print("alpha =", config["modele"]["alpha"], "| type :", type(config["modele"]["alpha"]).__name__)

{'graine': 42, 'donnees': {'entree': 'donnees_brutes.csv', 'sortie': 'donnees_propres.csv'}, 'modele': {'nom': 'ridge', 'alpha': 1.0}}
alpha = 1.0 | type : float


YAML se lit comme un dictionnaire Python : indentation = imbrication. `yaml.safe_load` (et jamais `yaml.load` sur un
fichier inconnu) le transforme en `dict`.

## 2. Trois étapes, trois fonctions, des fichiers entre elles

In [2]:
dossier = Path(tempfile.mkdtemp())
pd.DataFrame({"culture": ["maïs", "blé", "maïs", None], "rendement": [3.1, 2.4, None, 2.0]}).to_csv(dossier / "donnees_brutes.csv", index=False)


def preparer(cfg, dossier):
    df = pd.read_csv(dossier / cfg["donnees"]["entree"])
    propre = df.dropna()
    propre.to_csv(dossier / cfg["donnees"]["sortie"], index=False)
    return len(df), len(propre)


def entrainer(cfg, dossier):
    df = pd.read_csv(dossier / cfg["donnees"]["sortie"])  # lit la SORTIE de l'étape précédente
    return df.groupby("culture")["rendement"].mean().to_dict()


print("préparer :", preparer(config, dossier))
print("entraîner :", entrainer(config, dossier))

préparer : (4, 2)
entraîner : {'blé': 2.4, 'maïs': 3.1}


Chaque étape lit un fichier et en écrit un : on peut relancer « entraîner » sans refaire « préparer », regarder le
fichier intermédiaire, et tester chaque étape séparément.

## ⚠️ Piège : une erreur incompréhensible quand un fichier manque

In [3]:
cfg_faux = {**config, "donnees": {"entree": "donnees_brutes.csv", "sortie": "inexistant.csv"}}
try:
    entrainer(cfg_faux, dossier)
except FileNotFoundError as e:
    print("message brut de Python :", str(e)[:90], "…")

message brut de Python : [Errno 2] No such file or directory: '/tmp/tmphf5az5rg/inexistant.csv' …


Le message dit **quoi**, mais pas **quoi faire**. Dans un pipeline lancé par quelqu'un d'autre (ou par la CI), on
veut un message qui indique l'étape à lancer, et un code de sortie non nul pour que l'automate s'arrête.

In [4]:
def entrainer_robuste(cfg, dossier):
    chemin = dossier / cfg["donnees"]["sortie"]
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin.name} introuvable : lancer d'abord l'étape « préparer ».")
    return entrainer(cfg, dossier)


try:
    entrainer_robuste(cfg_faux, dossier)
except FileNotFoundError as e:
    print(e)

inexistant.csv introuvable : lancer d'abord l'étape « préparer ».


### ✍️ À toi de jouer

Écris une fonction `charger_config(texte)` qui lit le YAML et lève une `ValueError` claire si une des sections `graine`, `donnees` ou `modele` manque. Teste-la avec un YAML incomplet.

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
def charger_config(texte):
    cfg = yaml.safe_load(texte) or {}
    manquantes = [s for s in ("graine", "donnees", "modele") if s not in cfg]
    if manquantes:
        raise ValueError(f"Configuration incomplète : sections manquantes {manquantes}")
    return cfg

try:
    charger_config("graine: 1\n")
except ValueError as e:
    print(e)
```

**Pourquoi ?** on vérifie la configuration **au début**, avant de lancer un entraînement de plusieurs minutes qui planterait à la fin. Dans le projet, `agrireco.config.load_config` fait exactement cela, et la commande `agrireco` transforme ces erreurs attendues en un message lisible avec un code de sortie 1.
</details>

## ✅ À retenir

1. Étapes séparées (préparer ≠ entraîner ≠ évaluer), reliées par des fichiers.
2. Tous les réglages dans un seul fichier de configuration ; aucun chiffre magique dans le code.
3. `yaml.safe_load` pour lire un YAML ; la configuration est vérifiée dès le départ.
4. Une erreur attendue doit dire quoi faire, et rendre un code de sortie non nul.
5. Une commande par étape, plus une commande qui enchaîne tout (`agrireco all`).